# Representational similarity analysis

Visualizes `results/rsa.npz` from `scripts/run_analyses.py`. RDMs are 1 − Pearson r between unit timecourses (condition means, from frame2 onward). Corrections vs the legacy notebooks: trial averaging, strict-lower-triangle second-order correlation, and active-unit masking — see `illusion_rnn/analysis.py`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = np.load(ROOT / "results" / "rsa.npz", allow_pickle=False)
META = json.loads((ROOT / "results" / "meta.json").read_text())
VARIANTS = META["variants"]
DIRS = ["left", "middle", "right"]
print("active units:", META["n_active_units"], "of 2048")
print("accuracies:", META["accuracy"])

## 1. Unit-by-unit RDMs per direction

The same most-active units (shared across variants) — standard on top, outline below. Similar structure across the three directions within a variant means direction is encoded by trajectory geometry more than by which units co-fire.

In [ ]:
show_variants = [v for v in ("standard", "outline") if v in VARIANTS]
fig, axes = plt.subplots(len(show_variants), 3,
                         figsize=(11, 3.6 * len(show_variants)),
                         squeeze=False)
for row, variant in enumerate(show_variants):
    rdms = R[f"{variant}_unit_rdms"]
    for col in range(3):
        ax = axes[row][col]
        im = ax.imshow(rdms[col], vmin=0, vmax=2, cmap="viridis")
        ax.set_title(f"{variant} — {DIRS[col]}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
fig.colorbar(im, ax=axes[:, -1], shrink=0.75, label="1 − r")
fig.suptitle(f"Unit RDMs ({rdms.shape[1]} most-active shared units)")

## 2. Second-order RDMs: do directions share similarity structure?

Each cell compares two directions' full-network RDMs (active units only).

In [ ]:
fig, axes = plt.subplots(1, len(VARIANTS), figsize=(4.2 * len(VARIANTS), 3.6))
for ax, variant in zip(axes, VARIANTS):
    matrix = R[f"{variant}_condition_rdm"]
    im = ax.imshow(matrix, vmin=0, cmap="magma")
    ax.set_xticks(range(3))
    ax.set_xticklabels(DIRS, fontsize=8)
    ax.set_yticks(range(3))
    ax.set_yticklabels(DIRS, fontsize=8)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center",
                    va="center", color="w", fontsize=8)
    ax.set_title(f"{variant}")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("Condition (direction) second-order RDMs")
fig.tight_layout()

## 3. Cross-variant RSA: is outline TAM represented like standard TAM?

Second-order dissimilarity between all 9 (variant × direction) full-network RDMs. Block structure by variant (gridlines every 3) answers the headline question directly.

In [ ]:
keys = [str(k) for k in R["cross_keys"]]
matrix = R["cross_rdm"]
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(matrix, vmin=0, cmap="magma")
ax.set_xticks(range(len(keys)))
ax.set_xticklabels(keys, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(keys)))
ax.set_yticklabels(keys, fontsize=8)

# Block size and cut lines are derived from the actual variant/direction
# count rather than hardcoded to 3 variants -- with "basic" removed as a
# reported variant (it duplicates training data; see
# tests/test_stimuli_integrity.py) VARIANTS now has 2 entries, so the matrix
# is 6x6, not 9x9. Hardcoding block boundaries here previously produced a
# `matrix[0:3, 6:9]` slice on a 6x6 array -- an empty (3, 0) slice whose mean
# silently became NaN (with only a RuntimeWarning), rather than a loud
# failure.
n_dirs = len(DIRS)
n_variants = len(VARIANTS)
for cut in [n_dirs * i - 0.5 for i in range(1, n_variants)]:
    ax.axhline(cut, color="w", lw=1.5)
    ax.axvline(cut, color="w", lw=1.5)
fig.colorbar(im, label="1 − r (second-order)")
ax.set_title("Cross-variant representational similarity")
fig.tight_layout()
fig.savefig(ROOT / "figures" / "rsa_cross_variant.png", dpi=150,
            bbox_inches="tight")


def block_mean(row, col):
    """Mean of an n_dirs x n_dirs variant block; within-variant blocks
    exclude the structural zero diagonal (self-comparisons).

    ``row``/``col`` index into VARIANTS, not a hardcoded 3-variant layout --
    the block boundaries follow n_dirs and n_variants above, so this stays
    correct regardless of how many variants are reported.
    """
    r0, r1 = row * n_dirs, (row + 1) * n_dirs
    c0, c1 = col * n_dirs, (col + 1) * n_dirs
    block = matrix[r0:r1, c0:c1]
    if row == col:
        return float(block[~np.eye(n_dirs, dtype=bool)].mean())
    return float(block.mean())


print("within-standard (pairwise):", round(block_mean(0, 0), 3))
print("within-outline (pairwise):", round(block_mean(1, 1), 3))
print("standard-outline:", round(block_mean(0, 1), 3))

## Reading the matrix

The printed block means give the quantitative answer. The standard–outline block mean is much larger than the within-standard pairwise mean, and the within-outline mean is tiny — outline's three 'directions' produce essentially the same RDM, meaning the network extracts no direction information from outline stimuli. This is the representational counterpart of the behavioral pattern (standard ≈ 0.97, outline ≈ 0.10): the generalization failure lives in the representation itself, not in the readout.

Note: `basic`, a third stimulus directory shipped alongside `standard` and `outline`, is excluded from this comparison entirely — all 24 of its images are byte-identical to `standard` (`TAM_task`) images (21 share filenames, 3 are internal duplicates; see `tests/test_stimuli_integrity.py`). It is not a held-out variant, so it never appears as a row/column of the cross-variant RDM above.